# ScamShield Model Training 🛡️

Run this notebook in Google Colab to train the ScamShield Phishing Detection model. We use a Random Forest Classifier and export it to ONNX for production.

### Important Note on Datasets:
To train the model, the dataset **must** contain raw URLs. Datasets that only contain pre-calculated numbers (like `All.csv` or `Phishing.csv`) cannot be used because our API uses a custom `features.py` script to analyze live URLs. We will use the `PhiUSIIL_Phishing_URL_Dataset.csv` because it contains the actual URLs.

### Instructions:
1. Run the setup cell to install dependencies.
2. Run the Mount Drive cell to copy `features.py` and `PhiUSIIL_Phishing_URL_Dataset.csv` from your Google Drive into Colab.
3. Run the training cell to extract custom features, train the model, tune hyperparameters, and evaluate accuracy.
4. Download `model.onnx`, `feature_names.json`, and `metrics.json` to place in your `api/` folder.

In [ ]:
# 1. Install dependencies
!pip install -q scikit-learn skl2onnx onnxruntime pandas matplotlib seaborn tqdm bs4

In [ ]:
# 2. Mount Google Drive and Copy Files
import os
import shutil
from google.colab import drive

# Mount Drive (this will prompt you to authorize)
drive.mount('/content/drive')

# Copy files from Drive to local Colab environment
print("\nCopying files from Google Drive...")
shutil.copy('/content/drive/MyDrive/features.py', './features.py')

os.makedirs('./data', exist_ok=True)
dataset_path = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'
shutil.copy(dataset_path, './data/PhiUSIIL_Phishing_URL_Dataset.csv')

print("Files copied successfully!")

In [ ]:
# 3. Import Libraries
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve, average_precision_score
)
from sklearn.model_selection import train_test_split, GridSearchCV
from tqdm.auto import tqdm
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

warnings.filterwarnings("ignore", category=FutureWarning)
sys.path.insert(0, ".")
from features import extract_features, feature_names, FEATURE_NAMES

In [ ]:
# 4. Load Data & Extract Features
# Find the uploaded dataset
data_dir = Path("./data")
csv_files = list(data_dir.glob("*.csv")) + list(data_dir.glob("*.csv.gz"))
if not csv_files:
    raise FileNotFoundError("No dataset found in ./data/. Please upload one.")

DATA_FILE = csv_files[0]
MODEL_DIR = Path("./model")
MODEL_FILE = MODEL_DIR / "model.onnx"
FEATURE_NAMES_FILE = MODEL_DIR / "feature_names.json"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Loading dataset: {DATA_FILE}...")
if DATA_FILE.suffix == '.gz':
    df = pd.read_csv(DATA_FILE, compression="gzip")
else:
    df = pd.read_csv(DATA_FILE)

# Standardize column names (PhiUSIIL uses 'URL' and 'label')
if 'URL' in df.columns:
    df.rename(columns={'URL': 'url'}, inplace=True)
if 'label' not in df.columns and 'Label' in df.columns:
    df.rename(columns={'Label': 'label'}, inplace=True)

if 'url' not in df.columns or 'label' not in df.columns:
    raise ValueError(f"Dataset must contain a 'url' and 'label' column. Found: {df.columns.tolist()}")

# If the dataset is huge, sample it to save extraction time
if len(df) > 50000:
    print(f"Dataset is very large ({len(df)} rows). Sampling 50,000 for faster training...")
    df = df.sample(50000, random_state=42)

# Note: In PhiUSIIL, 1 is typically Legitimate and 0 is Phishing. ScamShield expects 1=Phishing, 0=Legitimate.
# Let's check the distribution. Usually there are more legitimate URLs.
if 'PhiUSIIL' in DATA_FILE.name:
    print("Adjusting labels for PhiUSIIL dataset (1=safe, 0=phish -> 0=safe, 1=phish)")
    df['label'] = df['label'].apply(lambda x: 1 if x == 0 else 0)

print(f"   Rows to process: {len(df)}")
print(f"   Phishing (1): {(df['label'] == 1).sum()}")
print(f"   Legitimate (0): {(df['label'] == 0).sum()}")

print("\n⚙️  Extracting custom ScamShield features from URLs...")
X_list, y_list = [], []
errors = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    try:
        url = str(row["url"])
        # We don't have HTML for external datasets usually, so pass empty strings
        html = str(row.get("html", "")) if pd.notna(row.get("html")) else ""
        content_available = bool(row.get("content_available", False))
        label = int(row["label"])
        
        feats = extract_features(url, html, content_available)
        X_list.append(feats)
        y_list.append(label)
    except Exception as e:
        errors += 1

if errors > 0:
    print(f"   ⚠️  {errors} rows had extraction errors (skipped)")

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)

print(f"\n📊 Feature matrix shape: {X.shape}")
assert X.shape[1] == len(FEATURE_NAMES), f"Feature count mismatch! Expected {len(FEATURE_NAMES)}, got {X.shape[1]}"

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"   Train: {X_train.shape[0]} samples")
print(f"   Test:  {X_test.shape[0]} samples")

In [ ]:
# 5. Train Model & Tune Hyperparameters
print("\n🌲 Training Random Forest Classifier with Grid Search...")
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [20, 30],
    'min_samples_split': [2, 5],
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

clf = grid_search.best_estimator_
print(f"\n🏆 Best Parameters: {grid_search.best_params_}")

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"\n📊 Test Set Results:")
print(f"   Accuracy: {acc:.4f}")
print(f"   F1 Score: {f1:.4f}")
print(f"   ROC AUC:  {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Legitimate", "Phishing"], yticklabels=["Legitimate", "Phishing"])
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

importances = clf.feature_importances_
indices = np.argsort(importances)[-15:]
axes[1].barh(range(15), importances[indices], color="#8B5CF6")
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([FEATURE_NAMES[i] for i in indices])
axes[1].set_title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Export to ONNX
from google.colab import files
print("\n📦 Exporting model to ONNX...")
initial_type = [("input", FloatTensorType([None, X.shape[1]]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type, target_opset=12, options={id(clf): {"zipmap": False}})

with open(MODEL_FILE, "wb") as f:
    f.write(onnx_model.SerializeToString())

with open(FEATURE_NAMES_FILE, "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

print(f"✅ Model exported to {MODEL_FILE}")
print(f"✅ Feature names saved to {FEATURE_NAMES_FILE}")

# 7. Download Files
print("\n📥 Downloading files to your machine...")
files.download(str(MODEL_FILE))
files.download(str(FEATURE_NAMES_FILE))